# 🏕️ Lagerplatz-Finder - Streamlit App

## ⚡ Quick Start (3 Schritte)

1. **Zelle 1:** Installation (1x)
2. **Zelle 2:** App erstellen
3. **Zelle 3:** App starten → Browser öffnet sich!

### ✨ Features:
- 🎚️ Interaktive Schieberegler
- 🗺️ Live-Karte mit Filterung
- 📊 Statistiken
- 📋 Daten-Export (CSV)
- 📱 Responsive Design

In [1]:
import geopandas as gpd
import time

print("🔍 DEBUG: App-Laden analysieren\n")

# Lade Daten
print("1️⃣ Lade GeoJSON...")
start = time.time()
gdf = gpd.read_file("output/geeignete_lagerflaechen_BL.geojson")
gdf = gdf.head(200)
load_time = time.time() - start
print(f"   ✅ Geladen in {load_time:.2f}s")
print(f"   Features: {len(gdf)}")

# Transformiere
print("\n2️⃣ Transformiere CRS...")
start = time.time()
gdf = gdf.to_crs("EPSG:4326")
transform_time = time.time() - start
print(f"   ✅ Transformiert in {transform_time:.2f}s")

# Erstelle Karte
print("\n3️⃣ Erstelle Folium-Karte...")
import folium
from streamlit_folium import st_folium

start = time.time()
bounds = gdf.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles="OpenStreetMap"
)

# Füge Features hinzu
for idx, row in gdf.iterrows():
    geom = row.geometry
    if geom.geom_type == "Polygon":
        coords = list(geom.exterior.coords)
        locations = [[lat, lon] for lon, lat in coords]
        folium.Polygon(
            locations=locations,
            color="#2d5016",
            fill=True,
            fillColor="#90EE90",
            fillOpacity=0.7,
            weight=2,
        ).add_to(m)

map_time = time.time() - start
print(f"   ✅ Karte erstellt in {map_time:.2f}s")

print(f"\n📊 GESAMT: {load_time + transform_time + map_time:.2f}s")
print(f"\n✅ Wenn alles < 5s ist, liegt Problem woanders!")

🔍 DEBUG: App-Laden analysieren

1️⃣ Lade GeoJSON...
   ✅ Geladen in 0.30s
   Features: 200

2️⃣ Transformiere CRS...
   ✅ Transformiert in 0.05s

3️⃣ Erstelle Folium-Karte...


2026-05-21 11:19:11.461 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


   ✅ Karte erstellt in 0.07s

📊 GESAMT: 0.42s

✅ Wenn alles < 5s ist, liegt Problem woanders!


## 📦 Zelle 1: Installation

In [1]:
import subprocess
import sys

print("🔧 Installiere Packages...\n")

packages = ['streamlit', 'geopandas', 'folium', 'streamlit-folium']

for package in packages:
    print(f"  📦 {package}...", end=" ", flush=True)
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', package],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("✅")

print("\n✅ Alle Packages installiert!")
print("\n👉 Führe jetzt Zelle 2 aus")

🔧 Installiere Packages...

  📦 streamlit... ✅
  📦 geopandas... ✅
  📦 folium... ✅
  📦 streamlit-folium... ✅

✅ Alle Packages installiert!

👉 Führe jetzt Zelle 2 aus


## ✍️ Zelle 2: Erstelle App-Datei

In [2]:
import os
from pathlib import Path

# App-Code
app_code = '''import streamlit as st
import geopandas as gpd
import folium
from streamlit_folium import st_folium
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ========================================================================
# CONFIG
# ========================================================================
st.set_page_config(
    page_title="🏕️ Lagerplatz-Finder",
    page_icon="🏕️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ========================================================================
# LOAD DATA
# ========================================================================
@st.cache_data
def load_data():
    """Lade Daten mit verschiedenen Pfaden"""
    paths = [
        "output/geeignete_lagerflaechen_BL.geojson",
        "./output/geeignete_lagerflaechen_BL.geojson",
        "geeignete_lagerflaechen_BL.geojson",
    ]
    
    for path in paths:
        if Path(path).exists():
            try:
                gdf = gpd.read_file(path)
                if gdf.crs != "EPSG:4326":
                    gdf = gdf.to_crs("EPSG:4326")
                return gdf, path
            except Exception as e:
                st.error(f"Fehler beim Laden: {e}")
                return None, None
    
    return None, None

# ========================================================================
# CREATE MAP
# ========================================================================
def create_map(gdf_data):
    """Erstelle Folium-Karte"""
    if gdf_data is None or len(gdf_data) == 0:
        return None
    
    bounds = gdf_data.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles="OpenStreetMap"
    )
    
    farben = {
        "forest": {"fill": "#2d5016", "outline": "#1a3009"},
        "meadow": {"fill": "#90EE90", "outline": "#228B22"},
        "other": {"fill": "#FFD700", "outline": "#FFA500"},
    }
    
    for idx, row in gdf_data.iterrows():
        try:
            geom = row.geometry
            if geom.geom_type != "Polygon":
                continue
            
            coords = list(geom.exterior.coords)
            locations = [[lat, lon] for lon, lat in coords]
            
            landuse = row.get("landuse", "other")
            farbe_config = farben.get(landuse, farben["other"])
            
            popup_html = f"""
            <div style="width: 280px; font-family: Arial; font-size: 12px;">
                <h4 style="color: #2d5016; margin-bottom: 8px;">🗺️ Lagerfläche #{row.get('lagerplatz_id', idx)}</h4>
                <table style="width: 100%; border-collapse: collapse;">
                    <tr style="background: #f0f0f0;">
                        <td style="padding: 6px; border: 1px solid #ccc;"><b>Fläche:</b></td>
                        <td style="padding: 6px; border: 1px solid #ccc; text-align: right;">{row.get('flaeche_ha', 0):.2f} ha</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px; border: 1px solid #ccc;"><b>Typ:</b></td>
                        <td style="padding: 6px; border: 1px solid #ccc;">{landuse}</td>
                    </tr>
                    <tr style="background: #f0f0f0;">
                        <td style="padding: 6px; border: 1px solid #ccc;"><b>Bewertung:</b></td>
                        <td style="padding: 6px; border: 1px solid #ccc;">{row.get('bewertung', '-')}</td>
                    </tr>
                </table>
                <hr style="margin: 8px 0;">
                <div style="font-size: 11px;">
                    <p style="margin: 5px 0;"><b>🏗️ Infrastruktur:</b></p>
                    <p style="margin: 3px 0;">🚜 Bauernhof: {row.get('dist_bauernhof_m', -1):.0f}m</p>
                    <p style="margin: 3px 0;">🚌 ÖV: {row.get('dist_oev_m', -1):.0f}m</p>
                </div>
            </div>
            """
            
            folium.Polygon(
                locations=locations,
                color=farbe_config["outline"],
                fill=True,
                fillColor=farbe_config["fill"],
                fillOpacity=0.7,
                weight=2,
                popup=folium.Popup(popup_html, max_width=350),
            ).add_to(m)
        except:
            continue
    
    folium.LayerControl().add_to(m)
    return m

# ========================================================================
# MAIN
# ========================================================================

st.markdown(
    "<h1 style=\"color: #2d5016; text-align: center;\">🏕️ Lagerplatz-Finder Baselland</h1>",
    unsafe_allow_html=True,
)

# Daten laden
gdf, path = load_data()

if gdf is None:
    st.error("❌ Datei nicht gefunden!")
    st.info("Erwartete Pfade: output/geeignete_lagerflaechen_BL.geojson")
    import sys
    sys.exit()

st.success(f"✅ {len(gdf)} Lagerplätze geladen von: {path}")

# SIDEBAR FILTER
st.sidebar.markdown("## 🎚️ Filter")

has_dist_cols = "dist_bauernhof_m" in gdf.columns and "dist_oev_m" in gdf.columns

if has_dist_cols:
    dist_bauernhof = st.sidebar.slider(
        "🚜 Bauernhöfe",
        min_value=0,
        max_value=3000,
        value=1000,
        step=100,
    )
    
    dist_oev = st.sidebar.slider(
        "🚌 ÖV-Haltestellen",
        min_value=0,
        max_value=3000,
        value=1000,
        step=100,
    )
    
    gdf_filtered = gdf[
        (gdf["dist_bauernhof_m"] <= dist_bauernhof)
        & (gdf["dist_oev_m"] <= dist_oev)
    ]
else:
    st.sidebar.warning("⚠️ Distanz-Spalten nicht vorhanden")
    gdf_filtered = gdf

# STATS
st.sidebar.markdown("---")
col1, col2 = st.sidebar.columns(2)
with col1:
    st.metric("📍 Gefiltert", len(gdf_filtered))
with col2:
    st.metric("📊 Gesamt", len(gdf))

if "flaeche_ha" in gdf_filtered.columns:
    st.sidebar.metric("🗺️ Fläche", f"{gdf_filtered['flaeche_ha'].sum():.0f} ha")

# TABS
tab1, tab2, tab3 = st.tabs(["🗺️ Karte", "📊 Statistiken", "📋 Daten"])

with tab1:
    if len(gdf_filtered) > 0:
        st.markdown(f"### 🗺️ Zeigt {len(gdf_filtered)} Lagerplätze")
        m = create_map(gdf_filtered)
        if m:
            st_folium(m, width=1200, height=600)
    else:
        st.error("❌ Keine Lagerplätze mit diesen Filtern")

with tab2:
    if len(gdf_filtered) > 0:
        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric("Lagerplätze", len(gdf_filtered))
        with col2:
            if "flaeche_ha" in gdf_filtered.columns:
                st.metric("Gesamtfläche", f"{gdf_filtered['flaeche_ha'].sum():.0f} ha")
        with col3:
            if "flaeche_ha" in gdf_filtered.columns:
                st.metric("Ø Fläche", f"{gdf_filtered['flaeche_ha'].mean():.2f} ha")
        
        if "landuse" in gdf_filtered.columns:
            st.markdown("### 🌍 Nach Landnutzung")
            st.write(gdf_filtered["landuse"].value_counts())
    else:
        st.warning("⚠️ Keine Daten")

with tab3:
    if len(gdf_filtered) > 0:
        cols_to_show = [
            col
            for col in gdf_filtered.columns
            if col != "geometry" and col != "index"
        ]
        df_show = gdf_filtered[cols_to_show].copy()
        st.dataframe(df_show, use_container_width=True, height=400)
        
        csv = df_show.to_csv(index=False)
        st.download_button(
            "📥 Download CSV",
            csv,
            "lagerflaechen.csv",
            "text/csv",
        )
    else:
        st.warning("⚠️ Keine Daten")

st.markdown("---")
st.markdown(
    '<div style="text-align: center; color: #999; font-size: 12px;">'
    "🏕️ Lagerplatz-Finder v1.0 | Baselland | Mai 2026"
    "</div>",
    unsafe_allow_html=True,
)
'''

# Speichere die App
with open('streamlit_app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✅ App-Datei erstellt: streamlit_app.py")
print("\n📁 Die App sucht nach:")
print("   • output/geeignete_lagerflaechen_BL.geojson")
print("   • ./output/geeignete_lagerflaechen_BL.geojson")
print("   • geeignete_lagerflaechen_BL.geojson")
print("\n👉 Führe jetzt Zelle 3 aus")

✅ App-Datei erstellt: streamlit_app.py

📁 Die App sucht nach:
   • output/geeignete_lagerflaechen_BL.geojson
   • ./output/geeignete_lagerflaechen_BL.geojson
   • geeignete_lagerflaechen_BL.geojson

👉 Führe jetzt Zelle 3 aus


## 🚀 Zelle 3: Starte die App

In [ ]:
import subprocess
import time
import webbrowser

print("""
╔════════════════════════════════════════════════════════════╗
║         🚀 STREAMLIT APP WIRD GESTARTET...                  ║
╚════════════════════════════════════════════════════════════╝

✨ Die App startet auf: http://localhost:8501

📌 Falls der Browser nicht öffnet:
   → Öffne manuell: http://localhost:8501

⏸️  Um die App zu stoppen:
   → Führe Zelle 4 aus
   ODER drücke Strg+C hier

💡 Die App nutzt: output/geeignete_lagerflaechen_BL.geojson
""")

try:
    # Versuche Browser zu öffnen
    print("🌐 Öffne Browser...\n")
    webbrowser.open('http://localhost:8501')
    time.sleep(2)
except:
    pass

# Starte Streamlit
try:
    subprocess.run(['streamlit', 'run', 'streamlit_app.py', '--logger.level=error'])
except KeyboardInterrupt:
    print("\n\n✋ App gestoppt!")


╔════════════════════════════════════════════════════════════╗
║         🚀 STREAMLIT APP WIRD GESTARTET...                  ║
╚════════════════════════════════════════════════════════════╝

✨ Die App startet auf: http://localhost:8501

📌 Falls der Browser nicht öffnet:
   → Öffne manuell: http://localhost:8501

⏸️  Um die App zu stoppen:
   → Führe Zelle 4 aus
   ODER drücke Strg+C hier

💡 Die App nutzt: output/geeignete_lagerflaechen_BL.geojson

🌐 Öffne Browser...



## ⏹️ Zelle 4: Stop-Befehle (falls nötig)

In [ ]:
import subprocess
import os
import signal

print("🛑 Versuche alle Streamlit-Prozesse zu stoppen...\n")

try:
    # Windows
    subprocess.run(['taskkill', '/F', '/IM', 'streamlit.exe'], stderr=subprocess.DEVNULL)
    print("✅ Windows-Prozesse gestoppt")
except:
    pass

try:
    # Linux/Mac
    subprocess.run(['pkill', '-f', 'streamlit'], stderr=subprocess.DEVNULL)
    print("✅ Linux/Mac-Prozesse gestoppt")
except:
    pass

print("\n✅ Alle Streamlit-Prozesse sollten gestoppt sein")
print("\n💡 Wenn die App immer noch läuft:")
print("   → Kernel neu starten (Kernel → Restart)")
print("   → ODER: Notebook-Tab schließen und neu öffnen")

## 📖 Hilfe

### ❓ Häufige Fragen

**Q: Die App startet nicht!**  
A: 
1. Stelle sicher, dass `output/geeignete_lagerflaechen_BL.geojson` existiert
2. Führe Zelle 1 (Installation) aus
3. Versuche Zelle 3 nochmal

**Q: Ich kann die App nicht stoppen!**  
A: Führe Zelle 4 aus (Stop-Befehle)

**Q: Browser öffnet nicht automatisch!**  
A: Öffne manuell: http://localhost:8501

**Q: Port 8501 ist bereits belegt!**  
A: Ändere in Zelle 3 die letzte Zeile zu:
```
subprocess.run(['streamlit', 'run', 'streamlit_app.py', '--server.port', '8502'])
```

**Q: Kann ich den Code ändern?**  
A: Ja! Ändere Zelle 2, führe sie aus, dann Zelle 3

### ⚙️ Wenn alles hängt

1. **Kernel neu starten:**
   - Kernel Menu → Restart Kernel
   - ODER: Strg+Shift+0

2. **Notebook-Tab schließen:**
   - Tab schließen
   - Wieder öffnen

3. **Terminal-Prozesse prüfen:**
   ```bash
   # Im Terminal:
   lsof -i :8501  # Zeigt was auf Port 8501 läuft
   kill -9 <PID>  # Stoppt den Prozess
   ```